# Isobutyl Alcohol

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
secbutylalcohol,74.122,3.3102,3.34415596,222.7977705,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
secbutylalcohol,H,secbutylalcohol,e,2000.0,0.03
"""

model = PCSAFT(["secbutylalcohol"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_secbutylalcohol.csv")
fix_line_endings("rhol_secbutylalcohol.csv")

Fixed: satp_secbutylalcohol.csv
Fixed: rhol_secbutylalcohol.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

805.1455286807915
473.34925052199856


In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_secbutylalcohol.csv",
        "satp_secbutylalcohol.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.9359906208893429


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2352.869393759923, 0.011422207806943991], PCSAFT{BasicIdeal, Float64}("secbutylalcohol"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_secbutylalcohol.csv",   my_saturation_p)


=== AAD: satp_secbutylalcohol.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
280.7300    662.000000    658.545454    0.5218  
281.0500    675.000000    675.271384    0.0402  
284.4800    878.000000    879.688481    0.1923  
287.9100    1136.000000   1137.064638   0.0937  
292.1600    1542.000000   1546.547042   0.2949  
292.2600    1552.000000   1557.569796   0.3589  
294.4500    1812.000000   1816.917134   0.2714  
298.0900    2347.000000   2332.236055   0.6291  
301.0200    2847.000000   2835.642994   0.3989  
303.9000    3412.000000   3420.249997   0.2418  
308.1500    4467.000000   4473.509694   0.1457  
311.6000    5515.000000   5524.589000   0.1739  
314.4100    6551.000000   6531.880138   0.2919  
AARD = 0.2811%


0.28110872814365717

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_secbutylalcohol.csv",   my_liquid_density)


=== AAD: rhol_secbutylalcohol.csv ===
Clapeyron Estimator  exp           calc          ARD%    


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


293.1500    806.700000    806.238122    0.0573  
298.1500    802.700000    801.921583    0.0970  
303.1500    798.400000    797.568699    0.1041  
308.1500    794.100000    793.174624    0.1165  
313.1500    789.800000    788.734545    0.1349  
318.1500    785.200000    784.243669    0.1218  
323.1500    780.600000    779.697209    0.1157  
AARD = 0.1067%


0.10674744739786879